# WasteWorth AI — Discover Hidden Value in Waste
### Complete Google Colab / Jupyter prototype

WasteWorth analyses a photo of discarded items, identifies visible objects, infers *possible* materials (including copper, aluminium, brass, steel and e-waste), and estimates a **range** of potential recovery value using editable rates.

**What this prototype does**
- Uses a vision API to analyse an uploaded image.
- Separates visible-object observations from uncertain material inferences.
- Calculates transparent low/high gross value ranges from estimated recoverable mass and editable material rates.
- Includes recovery suggestions, uncertainty notes, and safety flags.
- Runs as a Gradio web interface inside Colab or Jupyter.

**Honesty and safety limits**
A photograph cannot verify alloy, purity, hidden contents, mass, or live scrap prices. Weight ranges are rough model-assisted estimates, not measurements. Prices are example INR/kg values that must be edited for your location and buyer. Never dismantle batteries, capacitors, CRTs, or unknown electronics based on this app's output.

## 1. Install dependencies

Run this cell once. In Google Colab, restart the runtime only if Colab explicitly asks you to.

In [ ]:
%pip -q install openai gradio "pandas==2.2.3" pillow


## 2. Configure your vision API key

This notebook uses the OpenAI vision API (`gpt-4o-mini`). Create an API key in your OpenAI account and enter it in the app's password field. Do not paste the key into notebook source code or share a notebook containing a key.

If you do not have API access/billing enabled, the analysis call will fail; the rest of the project does not require a local model download.

## 3. Material rates and valuation logic

The default rates below are **demo placeholders, not current market quotes**. Edit them in the app before presenting results. The value is a scenario estimate, not a guaranteed resale offer.

In [ ]:
import json
import base64
import io
import os
import re
from typing import Any

import gradio as gr
import pandas as pd
from PIL import Image
from openai import OpenAI

DEFAULT_RATES = {
    "copper": 600.0,
    "aluminium": 120.0,
    "brass": 350.0,
    "steel_iron": 25.0,
    "stainless_steel": 70.0,
    "mixed_e_waste": 35.0,
    "circuit_board": 80.0,
    "plastic": 8.0,
    "cardboard_paper": 5.0,
    "glass": 2.0,
    "unknown_metal": 20.0,
    "other": 0.0
}

MATERIAL_LABELS = {
    "copper": "Copper",
    "aluminium": "Aluminium",
    "brass": "Brass",
    "steel_iron": "Steel / iron",
    "stainless_steel": "Stainless steel",
    "mixed_e_waste": "Mixed e-waste",
    "circuit_board": "Circuit board",
    "plastic": "Plastic",
    "cardboard_paper": "Cardboard / paper",
    "glass": "Glass",
    "unknown_metal": "Unknown metal",
    "other": "Other / no assigned rate"
}

ALLOWED_MATERIALS = set(DEFAULT_RATES)

## 4. Vision analysis and value calculation

The model is explicitly instructed not to claim that a visual appearance proves metal composition. It must return JSON with evidence and uncertainty. The weight range is a coarse visual estimate and should be replaced with a measured weight whenever possible.

In [ ]:
SYSTEM_PROMPT = """
You are WasteWorth AI, a cautious visual scrap-material triage assistant.
Inspect the supplied photograph and return ONLY valid JSON with this schema:
{
  "scene_summary": "short summary",
  "items": [
    {
      "object_name": "visible object name",
      "count": 1,
      "visible_evidence": "what is actually visible",
      "likely_materials": [
        {
          "material_key": "one of copper, aluminium, brass, steel_iron, stainless_steel, mixed_e_waste, circuit_board, plastic, cardboard_paper, glass, unknown_metal, other",
          "confidence": "high, medium, or low",
          "reason": "visual evidence and uncertainty"
        }
      ],
      "estimated_recoverable_mass_kg_low": 0.0,
      "estimated_recoverable_mass_kg_high": 0.0,
      "condition": "reusable, recoverable scrap, mixed/contaminated, damaged, or uncertain",
      "recommended_action": "safe next step",
      "safety_flags": ["..."]
    }
  ],
  "overall_cautions": ["..."]
}

Rules:
- Identify only objects actually visible; do not invent hidden components.
- A photo cannot establish exact alloy, purity, internal copper content, precious-metal content, or weight.
- Do not infer gold/silver/platinum value from a circuit board photo. Use circuit_board or mixed_e_waste unless the material is genuinely identifiable.
- Copper-coloured appearance is not proof of copper; yellow metal is not proof of brass; silver-coloured metal is not proof of aluminium.
- If material is uncertain, use unknown_metal or other and low confidence.
- Weight ranges must be conservative rough visual estimates. If scale/size is unknown, use 0 for both and explain that weighing is needed.
- Count visible items conservatively; do not count tiny components individually in a pile.
- Flag batteries, capacitors, chemical containers, sharp edges, and other hazards if visible. Never recommend unsafe dismantling.
- Keep the response concise, practical, and honest.
"""

def image_to_data_url(image):
    if image is None:
        raise ValueError("Please upload a waste photograph first.")
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    image = image.convert("RGB")
    image.thumbnail((1400, 1400))
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=88)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    return "data:image/jpeg;base64," + encoded

def parse_rates(rates_text):
    try:
        rates = json.loads(rates_text)
    except Exception as exc:
        raise ValueError("Rates must be valid JSON. Example: {\"copper\": 600, \"aluminium\": 120}") from exc
    clean = {}
    for key, value in rates.items():
        if key not in ALLOWED_MATERIALS:
            continue
        try:
            number = float(value)
            if number < 0:
                raise ValueError
            clean[key] = number
        except Exception:
            raise ValueError(f"Rate for {key} must be a non-negative number.")
    return clean

def call_vision_api(image, api_key, model_name):
    if not api_key or not api_key.strip():
        raise ValueError("Enter your vision API key in the API key field.")
    client = OpenAI(api_key=api_key.strip())
    data_url = image_to_data_url(image)
    response = client.chat.completions.create(
        model=(model_name or "gpt-4o-mini").strip(),
        temperature=0.1,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": "Analyse this waste photograph for visible recoverable objects and likely materials. Return the required JSON only."},
                {"type": "image_url", "image_url": {"url": data_url, "detail": "high"}}
            ]}
        ]
    )
    content = response.choices[0].message.content
    if not content:
        raise ValueError("The vision API returned an empty response.")
    result = json.loads(content)
    if not isinstance(result, dict):
        raise ValueError("Unexpected response format from the vision API.")
    result.setdefault("scene_summary", "Image analysed.")
    result.setdefault("items", [])
    result.setdefault("overall_cautions", [])
    return result

def safe_float(value, default=0.0):
    try:
        value = float(value)
        if value < 0:
            return default
        return value
    except (TypeError, ValueError):
        return default

def normalize_items(result):
    normalized = []
    for item in result.get("items", []):
        if not isinstance(item, dict):
            continue
        low = safe_float(item.get("estimated_recoverable_mass_kg_low"))
        high = safe_float(item.get("estimated_recoverable_mass_kg_high"))
        if high < low:
            low, high = high, low
        materials = []
        for mat in item.get("likely_materials", []):
            if not isinstance(mat, dict):
                continue
            key = str(mat.get("material_key", "other")).lower()
            if key not in ALLOWED_MATERIALS:
                key = "other"
            conf = str(mat.get("confidence", "low")).lower()
            if conf not in {"high", "medium", "low"}:
                conf = "low"
            materials.append({
                "material_key": key,
                "material_name": MATERIAL_LABELS[key],
                "confidence": conf,
                "reason": str(mat.get("reason", "No evidence provided."))
            })
        normalized.append({
            "object_name": str(item.get("object_name", "Unidentified object")),
            "count": max(1, int(safe_float(item.get("count", 1), 1))),
            "visible_evidence": str(item.get("visible_evidence", "")),
            "materials": materials or [{
                "material_key": "other",
                "material_name": MATERIAL_LABELS["other"],
                "confidence": "low",
                "reason": "Material could not be confidently inferred from the image."
            }],
            "mass_low": low,
            "mass_high": high,
            "condition": str(item.get("condition", "uncertain")),
            "recommended_action": str(item.get("recommended_action", "Inspect and verify material before recovery.")),
            "safety_flags": item.get("safety_flags", []) if isinstance(item.get("safety_flags", []), list) else []
        })
    return normalized

def build_report(result, rates):
    items = normalize_items(result)
    rows = []
    total_low = 0.0
    total_high = 0.0

    for item in items:
        # Valuation uses the first/primary material only to avoid double-counting
        # an object's full mass across several possible materials.
        primary = item["materials"][0]
        key = primary["material_key"]
        rate = rates.get(key, DEFAULT_RATES.get(key, 0.0))
        low_value = item["mass_low"] * rate
        high_value = item["mass_high"] * rate
        total_low += low_value
        total_high += high_value

        rows.append({
            "Object": item["object_name"],
            "Count": item["count"],
            "Primary material hypothesis": primary["material_name"],
            "Confidence": primary["confidence"].title(),
            "Mass low (kg)": round(item["mass_low"], 3),
            "Mass high (kg)": round(item["mass_high"], 3),
            "Rate (₹/kg)": round(rate, 2),
            "Gross value low (₹)": round(low_value, 2),
            "Gross value high (₹)": round(high_value, 2),
            "Condition": item["condition"],
            "Visible evidence": item["visible_evidence"],
            "Material reasoning": primary["reason"],
            "Recommended action": item["recommended_action"],
            "Safety flags": "; ".join(map(str, item["safety_flags"])) or "None noted"
        })

    df = pd.DataFrame(rows)
    return items, df, total_low, total_high

def render_markdown(result, items, total_low, total_high):
    lines = [
        "# WasteWorth recovery report",
        "",
        f"**Scene summary:** {result.get('scene_summary', 'No summary supplied.')}",
        "",
        f"## Indicative gross recovery value: ₹{total_low:,.0f}–₹{total_high:,.0f}",
        "",
        "> This is a rough scenario estimate based on model-suggested material and mass ranges and editable demo rates. It is not a quote, appraisal, or guaranteed resale value. Confirm material, weigh it, and obtain a local buyer quote.",
        "",
        "## Findings"
    ]
    if not items:
        lines.append("No items were returned. Try a clearer image with better lighting and fewer overlapping objects.")
    for idx, item in enumerate(items, 1):
        primary = item["materials"][0]
        lines.extend([
            f"### {idx}. {item['object_name']} (count: {item['count']})",
            f"- **Visible evidence:** {item['visible_evidence'] or 'Not specified'}",
            f"- **Primary material hypothesis:** {primary['material_name']} — **{primary['confidence']} confidence**",
            f"- **Why:** {primary['reason']}",
            f"- **Other possible materials:** " + ", ".join(
                f"{m['material_name']} ({m['confidence']})" for m in item["materials"][1:]
            ) if len(item["materials"]) > 1 else "- **Other possible materials:** None listed",
            f"- **Estimated recoverable mass:** {item['mass_low']:.3f}–{item['mass_high']:.3f} kg (visual estimate; weigh to verify)",
            f"- **Condition:** {item['condition']}",
            f"- **Suggested next step:** {item['recommended_action']}",
            f"- **Safety flags:** {', '.join(map(str, item['safety_flags'])) if item['safety_flags'] else 'None noted'}",
            ""
        ])
    cautions = result.get("overall_cautions", [])
    lines.extend(["## Overall cautions"])
    lines.extend([f"- {c}" for c in cautions] if cautions else ["- Visual analysis cannot verify composition, purity, hidden components, weight, or current buyer rates."])
    lines.extend([
        "",
        "## Safety",
        "- Do not open or dismantle batteries, capacitors, CRTs, sealed units, or unknown electronic equipment based on this report.",
        "- Treat sharp, contaminated, chemically exposed, or unidentified items as potentially hazardous.",
        "- Use trained handlers and appropriate PPE for inspection and recovery."
    ])
    return "\n".join(lines)

def analyze_waste(image, api_key, model_name, rates_text):
    try:
        rates = parse_rates(rates_text)
        result = call_vision_api(image, api_key, model_name)
        items, df, total_low, total_high = build_report(result, rates)
        report = render_markdown(result, items, total_low, total_high)
        if df.empty:
            df = pd.DataFrame(columns=[
                "Object", "Count", "Primary material hypothesis", "Confidence",
                "Mass low (kg)", "Mass high (kg)", "Rate (₹/kg)",
                "Gross value low (₹)", "Gross value high (₹)", "Condition",
                "Visible evidence", "Material reasoning", "Recommended action", "Safety flags"
            ])
        return report, df, json.dumps(result, indent=2, ensure_ascii=False)
    except Exception as exc:
        message = f"## Analysis could not be completed\n\n`{type(exc).__name__}: {exc}`\n\nCheck your API key, API access, model name, internet connection, and rates JSON."
        empty = pd.DataFrame()
        return message, empty, "{}"

## 5. Launch the interactive app

Paste your API key into the password field after launching. The key is passed to the API client for the current request and is not written to a file by this notebook. Avoid sharing screenshots containing your key.

In [ ]:
rates_json = json.dumps(DEFAULT_RATES, indent=2)

with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="rose"),
               title="WasteWorth AI — Hidden Value Discovery") as demo:
    gr.Markdown("""
    # ♻️ WasteWorth AI
    ### Discover hidden value in discarded materials
    Upload a photo of scrap, electronics, an appliance, or a mixed waste pile. Review the evidence and uncertainty before acting.
    """)
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload waste photograph", height=360)
            api_key_input = gr.Textbox(label="OpenAI API key", type="password", placeholder="Enter key for this session")
            model_input = gr.Textbox(label="Vision model", value="gpt-4o-mini")
            rates_input = gr.Textbox(label="Editable example rates (INR per kg, JSON)", value=rates_json, lines=14)
            analyze_btn = gr.Button("Analyse image & estimate recovery value", variant="primary")
            gr.Markdown("""
            **Rate note:** Values are placeholders, not live scrap rates. Edit them using local buyer quotes.
            """)
        with gr.Column(scale=1):
            report_output = gr.Markdown(label="Recovery report")
            table_output = gr.Dataframe(label="Item-by-item estimate", interactive=False, wrap=True)
    raw_output = gr.Code(label="Raw vision response (for debugging)", language="json", lines=12)
    analyze_btn.click(
        fn=analyze_waste,
        inputs=[image_input, api_key_input, model_input, rates_input],
        outputs=[report_output, table_output, raw_output]
    )
    gr.Markdown("""
    ---
    **Safety:** AI material identification is a screening aid, not a chemical or metallurgical test. Verify materials physically using safe, qualified procedures. Do not dismantle hazardous electronics.
    """)

demo.launch(share=True, debug=True)

## 6. Test plan for the demo

Try three photographs with good lighting and clearly visible items:
1. An old electric motor or fan (possible copper windings, steel, aluminium).
2. A pile of wires and metal fittings (possible copper, brass, other metals).
3. An electronic circuit board with cables (e-waste; do not assume precious-metal content).

For a stronger and more honest demonstration:
- Weigh one or two items on a scale and compare the real mass with the model's visual estimate.
- Replace example rates with recent quotes from local scrap buyers.
- Show at least one uncertain result and explain why human verification is needed.
- If the API returns an unsupported or incorrect material, correct the classification and explain that the model is a triage assistant, not a certified assay.

## 7. Troubleshooting

- **401 / authentication error:** Confirm the API key is valid and has API access.
- **429 / quota error:** Check API billing or usage limits.
- **Model not found:** Use a vision-capable model enabled for your API account.
- **Invalid JSON:** The app asks for JSON mode; retry with a clearer image or a supported vision model.
- **Poor metal identification:** Photograph items separately, use brighter light, show both sides, and add a specialised material classifier or verified human review.
- **Unrealistic valuation:** Treat the mass estimate as unknown and weigh the object. The app's valuation depends directly on the mass and rate inputs.